In [6]:
%load_ext autoreload
%autoreload 2

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import sys
# sys.path.insert(0, "/home/belle/zhangboy/B2SW/2025_VirginiaTech/sysvar/src/")
sys.path.append('/home/belle/zhangboy/inclusive_R_D/')
import utilities as util

training_variables = util.training_variables
relevant_vars = training_variables + ['target','training_weight','Ecms','B0_CMS_roeP_my_mask', 'B0_dr']
columns = util.all_relevant_variables
offline_cut = util.offline_cut
lgb_tight = util.lgb_tight
lgb_loose = util.lgb_loose
lgb_comb = util.lgb_comb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
 # load data
sig_mc15ri = uproot.concatenate(['/home/belle/zhangboy/inclusive_R_D/Samples/sig_MC15ri_quaxo_e_1/sigDDst.root:B0'],
                                cut=offline_cut,library="np",
                                filter_branch=lambda branch: branch.name in columns)
df_sig = pd.DataFrame(sig_mc15ri)
df_sig['ell_BFbrems_mcPDG'] = df_sig['ell_mcPDG']
sig_samples=util.classify_mc_dict(df_sig, 'e', template=False)
correct_signal = sig_samples[r'$D\tau\nu$']
print('signal sample size', len(correct_signal) )


generic_mc16rd = uproot.concatenate(['/home/belle/zhangboy/inclusive_R_D/Samples/MC16rd/4S_run1_deimos_1/*.root:B0_e'],
                                    cut=offline_cut, library="np",
                                    filter_branch=lambda branch: branch.name in columns)
df_generic = pd.DataFrame(generic_mc16rd)
generic_samples=util.classify_mc_dict(df_generic, 'e', template=False)
fakeD = generic_samples['bkg_fakeD']
continuum = generic_samples['bkg_continuum']
combinatorial = generic_samples['bkg_combinatorial']
print('fakeD sample size', len(fakeD))
print('continuum sample size', len(continuum))
print('combinatorial sample size', len(combinatorial))



# data_4Soffres = uproot.concatenate(['Samples/Data/e_channel/proc16_4Soffres_deimos_1.root:B0'],
#                       library="np",cut = cut,filter_branch=lambda branch: branch.name in relevant_vars)
# df_continuum = pd.DataFrame(data_4Soffres).drop('B0_roeMbc_my_mask', axis=1) 
# df_continuum.eval('B0_roeMbc_my_mask = ( (10.58/2)**2 - (B0_CMS_roeP_my_mask*10.58/Ecms)**2 )**0.5', inplace=True)
# df_continuum['weight']=2
# df_continuum['target']=3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
signal sample size 53069
fakeD sample size 15815120
continuum sample size 734906
combinatorial sample size 739879


In [8]:
correct_signal['training_weight']=15
correct_signal['target']=0
fakeD_sub = fakeD.sample(n=750000, random_state=0)
fakeD_sub['training_weight']=1
fakeD_sub['target']=1
continuum['training_weight']=1
continuum['target']=2
combinatorial['training_weight']=1
combinatorial['target']=3

df_all_classes = pd.concat([correct_signal, fakeD_sub, continuum, combinatorial])
with uproot.recreate("../Samples/BDT_training_sample.root") as file:
    file["all_classes"] = df_all_classes[relevant_vars]